In [1]:
# connect to google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

"""
Data Augmentation using Danish BERT

"""

!pip install textattack -q

import pandas as pd
from textattack.transformations import WordSwapMaskedLM
from textattack.constraints.semantics import WordEmbeddingDistance
from textattack.constraints.pre_transformation import RepeatModification, StopwordModification
from textattack.augmentation import Augmenter
from datetime import datetime
import os
import glob # Added for finding lock files

# Configuration
TEXT_COLUMN = 'ArticleText'
LABEL_COLUMN = 'has_claim'
PCT_WORDS_TO_SWAP = 0.1
TRANSFORMATIONS_PER_EXAMPLE = 2
MAX_TOKENS = 512
SAVE_EVERY = 20

# Paths
DRIVE_BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/misleading-climate-claims-dk/data/processed/cycle3'
INPUT_PATH = f'{DRIVE_BASE_PATH}/train_manual_2103.parquet'

def truncate_text(text, max_tokens=512):
    tokens = text.split()
    return ' '.join(tokens[:max_tokens]) if len(tokens) > max_tokens else text

print("="*60)
print("BERT-BASED AUGMENTATION")
print("="*60)

# Load data
train_df = pd.read_parquet(INPUT_PATH)
train_df['is_augmented'] = False
train_df[TEXT_COLUMN] = train_df[TEXT_COLUMN].astype(str)

# Truncate
train_df[TEXT_COLUMN] = train_df[TEXT_COLUMN].apply(lambda x: truncate_text(x, MAX_TOKENS))

# Get positive examples
positive_df = train_df[train_df[LABEL_COLUMN] == 1].copy().reset_index(drop=True)
print(f"Total positive examples to augment: {len(positive_df):,}\n")

# ============================================================================
# CHECK FOR EXISTING AUGMENTATION RUN
# ============================================================================

temp_folders = sorted([f for f in os.listdir(DRIVE_BASE_PATH)
                       if f.startswith('augmentation_temp_bert_')])

if temp_folders:
    print("Found existing augmentation run(s):")
    for folder in temp_folders:
        print(f"  - {folder}")

    # Use most recent
    temp_dir = f'{DRIVE_BASE_PATH}/{temp_folders[-1]}'
    print(f"\nResuming: {temp_folders[-1]}")

    # Count existing batches
    existing_batches = sorted([f for f in os.listdir(temp_dir)
                               if f.startswith('batch_')])

    if existing_batches:
        start_idx = len(existing_batches) * SAVE_EVERY
        batch_count = len(existing_batches)
        print(f"Found {len(existing_batches)} existing batches")
        print(f"Resuming from example {start_idx:,} (batch {batch_count})\n")
    else:
        start_idx = 0
        batch_count = 0
        print("No batches found, starting fresh\n")
else:
    # Create new temp directory
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    temp_dir = f'{DRIVE_BASE_PATH}/augmentation_temp_bert_{timestamp}'
    os.makedirs(temp_dir, exist_ok=True)
    start_idx = 0
    batch_count = 0
    print(f"Created new temp directory: {temp_dir}\n")

# Check if already complete
if start_idx >= len(positive_df):
    print("="*60)
    print("\u2713 AUGMENTATION ALREADY COMPLETE!")
    print("="*60)
    print(f"All {len(positive_df):,} examples have been processed.")
    print("\nRun the COMBINING section below to create final dataset.")

else:
    # Continue from where we left off
    remaining_df = positive_df.iloc[start_idx:].copy()

    print("="*60)
    print("AUGMENTING")
    print("="*60)
    print(f"Remaining: {len(remaining_df):,} examples")
    print(f"Starting from index: {start_idx:,}\n")

    # Ensure no lingering file locks from previous runs
    # This addresses the 'RuntimeError: Deadlock' caused by textattack's filelock usage.
    lock_files = glob.glob('/root/.cache/textattack/**/*.lock', recursive=True)
    for lock_file in lock_files:
        try:
            os.remove(lock_file)
            print(f"Removed lingering textattack lock file: {lock_file}")
        except OSError as e:
            print(f"Error removing lock file {lock_file}: {e}")

    # Create BERT-based augmenter
    print("Loading Danish BERT augmenter")
    augmenter = Augmenter(
        transformation=WordSwapMaskedLM(
            method="bae",
            masked_language_model="Maltehb/danish-bert-botxo",
            max_candidates=10,
            min_confidence=0.3
        ),
        constraints=[
            RepeatModification(),
            StopwordModification(),
            WordEmbeddingDistance(min_cos_sim=0.8)
        ],
        pct_words_to_swap=PCT_WORDS_TO_SWAP,
        transformations_per_example=TRANSFORMATIONS_PER_EXAMPLE
    )
    print("Augmenter loaded\n")

    augmented_rows = []
    failed_count = 0

    for idx, row in remaining_df.iterrows():
        try:
            text = str(row[TEXT_COLUMN]).strip()

            if len(text) < 5:
                failed_count += 1
                continue

            augmented_texts = augmenter.augment(text)

            if not isinstance(augmented_texts, list):
                augmented_texts = [augmented_texts]

            for aug_text in augmented_texts:
                new_row = row.copy()
                new_row[TEXT_COLUMN] = aug_text
                new_row['is_augmented'] = True
                augmented_rows.append(new_row)

        except Exception as e:
            failed_count += 1
            if failed_count <= 5:
                print(f"Failed: {e}")

        # Progress
        processed = len(augmented_rows) + failed_count
        if processed % 25 == 0:
            total_processed = start_idx + processed
            print(f"  Processed {total_processed}/{len(positive_df)}... ({len(augmented_rows):,} augmented in this session)")

        # Save batch
        if processed % SAVE_EVERY == 0 and augmented_rows:
            batch_df = pd.DataFrame(augmented_rows)
            batch_df.to_parquet(f'{temp_dir}/batch_{batch_count:04d}.parquet', index=False)
            print(f"Saved batch_{batch_count:04d}.parquet")
            augmented_rows = []
            batch_count += 1

    # Save final batch
    if augmented_rows:
        batch_df = pd.DataFrame(augmented_rows)
        batch_df.to_parquet(f'{temp_dir}/batch_{batch_count:04d}.parquet', index=False)
        print(f"Saved batch_{batch_count:04d}.parquet")

    print("\n" + "="*60)
    print("SESSION COMPLETE!")
    print("="*60)
    print(f"Failed in this session: {failed_count}/{len(remaining_df)}")
    print(f"Temp directory: {temp_dir}")
    print(f"\nTo continue or finish, just run this cell again!")

# ============================================================================
# COMBINING BATCHES (Always runs to show current progress)
# ============================================================================

print("\n" + "="*60)
print("CURRENT PROGRESS")
print("="*60)

# Check if temp_dir exists before listing its contents
if os.path.exists(temp_dir):
    batch_files = sorted([f for f in os.listdir(temp_dir) if f.startswith('batch_')])
else:
    batch_files = [] # If temp_dir doesn't exist, there are no batches
print(f"Total batches saved: {len(batch_files)}")

if len(batch_files) * SAVE_EVERY >= len(positive_df):
    print("All examples have been augmented!")
    print("\nCreating final combined dataset...")

    # Load all batches
    all_augmented = []
    for batch_file in batch_files:
        batch_df = pd.read_parquet(f'{temp_dir}/{batch_file}')
        all_augmented.append(batch_df)

    augmented_df = pd.concat(all_augmented, ignore_index=True)

    # Combine original + augmented
    combined_df = pd.concat([train_df, augmented_df], ignore_index=True)

    # Save final dataset
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_path = f'{DRIVE_BASE_PATH}/train_augmented_{timestamp}.parquet'
    combined_df.to_parquet(output_path, index=False)

    # Sample augmentations
    print("\n" + "="*60)
    print("SAMPLE AUGMENTATIONS")
    print("="*60)
    print("Showing 3 random augmented examples:\n")

    sample_augmented = augmented_df.sample(min(3, len(augmented_df)))

    for idx, (_, row) in enumerate(sample_augmented.iterrows(), 1):
        print(f"Example {idx}:")
        print(f"  {row[TEXT_COLUMN][:250]}...")
        print()

    # Final statistics
    print("="*60)
    print("RESULTS")
    print("="*60)
    print(f"Original: {len(train_df):,} ({train_df[LABEL_COLUMN].sum():,} positive)")
    print(f"Augmented: {len(augmented_df):,} (all positive)")
    print(f"Combined: {len(combined_df):,} ({combined_df[LABEL_COLUMN].sum():,} positive)")

    final_positive = combined_df[LABEL_COLUMN].sum()
    final_negative = len(combined_df) - final_positive
    print(f"\nFinal class distribution:")
    print(f"  Positive: {final_positive:,} ({final_positive/len(combined_df):.1%})")
    print(f"  Negative: {final_negative:,} ({final_negative/len(combined_df):.1%})")
    print(f"  New imbalance ratio: {final_negative/final_positive:.2f}:1")

    print(f"\n{'='*60}")
    print(f"\u2713 AUGMENTATION COMPLETE")
    print(f"{'='*60}")
    print(f"Saved to: {output_path}")
    print(f"Temp files: {temp_dir}/")
    print(f"\nYou can now delete the temp folder and use the augmented dataset!")

else:
    # Only calculate expected_batches if positive_df is not empty to avoid division by zero
    if len(positive_df) > 0:
        expected_batches = (len(positive_df) + SAVE_EVERY - 1) // SAVE_EVERY
        print(f"Progress: {len(batch_files)}/{expected_batches} batches")
    else:
        print("No positive examples to augment.")
    print(f"\nRun this cell again to continue augmentation!")


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 18.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 64.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 445.7/445.7 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.2/63.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

textattack: Updating TextAttack package dependencies.
textattack: Downloading NLTK required packages.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package omw to /root/nltk_data...
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape s

BERT-BASED AUGMENTATION (RESUMABLE)
Total positive examples to augment: 393

Created new temp directory: /content/drive/MyDrive/Colab Notebooks/misleading-climate-claims-dk/data/processed/cycle3/augmentation_temp_bert_20260521_132705

AUGMENTING
Remaining: 393 examples
Starting from index: 0

Loading Danish BERT augmenter (this may take a minute)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

textattack: Downloading https://textattack.s3.amazonaws.com/word_embeddings/paragramcf.

  0%|          | 0.00/481M [00:00<?, ?B/s]
  0%|          | 122k/481M [00:00<13:01, 616kB/s]
  0%|          | 184k/481M [00:00<28:52, 278kB/s]
  0%|          | 379k/481M [00:00<12:42, 631kB/s]
  0%|          | 973k/481M [00:00<04:17, 1.86MB/s]
  1%|          | 2.81M/481M [00:00<01:19, 5.99MB/s]
  1%|▏         | 7.21M/481M [00:00<00:29, 15.9MB/s]
  2%|▏         | 11.1M/481M [00:01<00:21, 22.3MB/s]
  3%|▎         | 15.2M/481M [00:01<00:16, 27.5MB/s]
  4%|▍         | 19.8M/481M [00:01<00:14, 32.7MB/s]
  5%|▌         | 24.2M/481M [00:01<00:12, 36.0MB/s]
  6%|▌         | 28.6M/481M [00:01<00:11, 38.2MB/s]
  7%|▋         | 32.6M/481M [00:01<00:11, 38.5MB/s]
  8%|▊         | 36.6M/481M [00:01<00:11, 39.2MB/s]
  8%|▊         | 40.6M/481M [00:01<00:11, 37.5MB/s]
  9%|▉         | 44.5M/481M [00:01<00:12, 35.4MB/s]
 10%|▉         | 48.1M/481M [00:02<00:14, 30.0MB/s]
 11%|█         | 51.3M/481M [00:02<00:19, 2

Augmenter loaded

